# 01 Build Loan-Month Panel

**Purpose:** Join origination traits to the monthly history and flag first default / first prepayment.

**Steps**
- Join on Loan Sequence Number
- Derive vintage quarter and MOB
- Default flag (90+ DPD / RA / loss ZBC), absorbing
- Prepay flag (ZBC 01)
- Loan-level first event table
- Sanity checks

**Read first:** Bellini 2.2 (default definition)

**Done when:** panel and loan-level event table for 2007 saved to `data/processed/`, checks pass.

**Year in this pass:** 2007 only. Functions that work get moved into `src/freddie/`.

## Setup

In [1]:
# Imports + Notebook Setup

import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.append(r"C:\Dev\CreditRiskLearning\Projects\Freddie Mac End-to-End Project\src")
from freddie.config import RAW_DIR, INTERIM_DIR, DEFAULT_ZBC, PREPAY_ZBC, CENSOR_ZBC

from freddie.load import load_orig, load_perf, sanity_check
from freddie.panel import add_mob, vintage_quarter, default_flagger, default_flag_absorbing, prepay_flag, censor_flag, summarise_loans

years = [2006, 2007, 2010, 2016]

In [2]:
orig = {year: load_orig(year, save=False) for year in years}

2006 orig: 50,000 rows x 31 cols
2007 orig: 50,000 rows x 31 cols
2010 orig: 50,000 rows x 31 cols
2016 orig: 50,000 rows x 31 cols


In [3]:
perf= {year: load_perf(year, save=False) for year in years}

In [4]:
# Add Vintage Quarter

for year in years: 
    vintage_quarter(perf[year])
    


In [5]:
# Add MOB
# loan_age resets after a loan modification (User Guide, LOAN AGE), so it can't be the clock.
# MOB counts months from the first payment date to the reporting period, +1 so it matches loan_age on unmodified loans.

for year in years: 
    add_mob(perf[year], orig[year])
    


In [6]:
# Add 90+ DPD Flag
for year in years: 
    default_flagger(perf[year])
    




In [10]:
# Default absorbing function
for year in years: 
    perf[year] = default_flag_absorbing(perf[year])
    


In [11]:
perf[2006].sample(10)

,loan_id,period,current_upb,current_dlq_status,loan_age,zero_balance_code,zero_balance_date,vintage,mob,default_flag,default_flag_absorbing
166379,F06Q10071066,200912,82533.47,00,46,NaN,<NA>,2006Q1,46,0,0
1587700,F06Q20380081,200803,63606.38,00,20,NaN,<NA>,2006Q2,20,0,0
2060399,F06Q30232251,200909,251704.40,00,35,NaN,<NA>,2006Q3,35,0,0
2428453,F06Q40022465,201305,199731.35,00,12,NaN,<NA>,2006Q4,78,0,1
1043486,F06Q20091585,200609,256000.00,00,4,NaN,<NA>,2006Q2,4,0,0
2631832,F06Q40129448,201008,206934.70,00,45,NaN,<NA>,2006Q4,45,0,0
152066,F06Q10062781,200801,222728.44,00,23,NaN,<NA>,2006Q1,23,0,0
1001771,F06Q20068862,200611,211000.00,00,5,NaN,<NA>,2006Q2,5,0,0
841078,F06Q10361820,200709,125918.48,00,18,NaN,<NA>,2006Q1,18,0,0
3199885,F06Q40431273,201501,94668.28,00,98,NaN,<NA>,2006Q4,98,0,0


In [12]:
# Add Prepay ZBD Flag
for year in years: 
    prepay_flag(perf[year])
    

In [13]:
# Censor Flag

for year in years: 
    censor_flag(perf[year])
    


In [16]:
# Collapse into Loan Level First Event Table

loan_table = {}

for year in years: 
    loan_table[year] = summarise_loans(perf[year])
    


**Observation (Written by AI).** 142 loans showed both a prepay and a default. All 142 had a default *after* the payoff row when sorted by `loan_age`. Cause: the User Guide says `loan_age` restarts from the modification first payment date for modified loans. In 2006, 2,651 loans (about 5%) have at least one reset and 133,321 rows share a `loan_age` with another row of the same loan, so sorting by `loan_age` put their months out of time order.

**Decision.** Use `mob` (months from `first_payment_date` to `period`, +1) as the clock for every sort and event month. It never resets and equals `loan_age` on unmodified loans. With `mob`, the overlap is 0 and outcome counts are unchanged (prepaid 41,951, default 7,215, open 744, censored 90), but 1,107 of the 7,215 defaults (15%) move to a different default month (median 26 months later). `loan_age` is kept in the panel for reference only.
